# APEX AI — Phase 6: Autonomous Technical Tool-Maker Fine-Tuning (QLoRA on Free Colab GPU)

This notebook demonstrates how to fine-tune **Llama-3-8B-Instruct** using **QLoRA (4-bit Low-Rank Adaptation)** on Google Colab's free T4 GPU to transform it into a **Senior F1 Technical Director & Autonomous Tool Maker**.

### What this accomplishes:
1. **Technical F1 Domain Specialization**: Forces the model to think and speak in terms of aerodynamic balance, tyre degradation slopes, lateral load dynamics, and telemetry deltas.
2. **Dynamic Tool Generation (LATM)**: When presented with a complex, novel query, the model writes custom Python/SQL code to query `f1.db` on-the-fly, executes it, and synthesizes the mathematical results.

---

## 1. Install Unsloth / Hugging Face QLoRA Libraries
(Unsloth accelerates fine-tuning 2-5x and reduces VRAM usage to fit easily in Colab's free 16GB T4 GPU).

In [ ]:
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes triton

## 2. Load 4-bit Quantized Llama-3-8B with Fast LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detection (Float16 on T4)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA adapters to attention and MLP layers
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

## 3. Clone Repository & Load Technical Instruction Dataset

In [ ]:
!git clone https://github.com/bhxvxsh/yeah-def-my-nyc-project.git repo

from datasets import load_dataset
dataset = load_dataset("json", data_files="repo/APEX_AI/03_Training_Dataset/f1_technical_alpaca.json", split="train")

prompt_template = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = prompt_template.format(instruction, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

## 4. Train the Model (Takes ~15-20 minutes on free T4 GPU)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

## 5. Test Autonomous Tool Generation on a Complex Query

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Evaluate the net position delta and mechanical recovery efficiency of Sergio Perez during the 2023 season when starting outside the top 5.

### Response:
"""

inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])